# Pipeline para la clasificación de los proyectos con la taxonomía PAE, usando un enfoque basado en similitud del coseno y embeddings.


Baseline fuerte: Embeddings + Top-k por similitud

Qué haces

Representas cada categoría PAE con un texto: category_label + " " + text (descripción).

Calculas embeddings para:

tax_emb (todas las categorías)

proj_emb (cada proyecto, usando text)

Calculas similitud coseno S = proj_emb · tax_emb^T

Para cada proyecto asignas:

Top-1 (clasificación simple)

o Top-k (p.ej. Top-3) con scores

Por qué es buena primera opción

No necesitas etiquetas.

Es interpretable (puedes mostrar top-k candidatos + score).

Escala bien.

Qué modelo usar

all-mpnet-base-v2 funciona, pero para retrieval suele ir aún mejor:

intfloat/e5-base-v2 o intfloat/e5-large-v2 (si GPU)

(con prompt estilo: "query: <texto proyecto>" y "passage: <texto categoría>")

## 1. Cargar corpus y taxonomía

In [12]:
# Necesario montar nuestra unidad de google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os
import pandas as pd

os.chdir("/content/drive/MyDrive/PROYECTOS/PAE")
print("DIRECTORIO DE TRABAJO:", os.getcwd())
#os.listdir()

FILE_CORPUS = "data/pae_aero_corpus.csv"
print("FICHERO DEL CORPUS:", FILE_CORPUS)

df_proyectos = pd.read_csv(FILE_CORPUS, encoding="utf-8")
print("Shape:", df_proyectos.shape)

if "Nivel_1" in list(df_proyectos.columns):
    df_proyectos.drop(columns=["Nivel_1"], inplace=True)
print("Columnas:", list(df_proyectos.columns))

display(df_proyectos.head(2))

texts = df_proyectos["text"]
print("Total proyectos:", len(texts))


DIRECTORIO DE TRABAJO: /content/drive/MyDrive/PROYECTOS/PAE
FICHERO DEL CORPUS: data/pae_aero_corpus.csv
Shape: (733, 3)
Columnas: ['title', 'description', 'text']


,title,description,text
0,Tilt Rotor ATM Integrated Validation of Enviro...,"""Cleansky-Green Rotor Craft sub-project 5 is a...",Tilt Rotor ATM Integrated Validation of Enviro...
1,GNSS-based ATM for Rotorcraft to Decrease Emis...,The GARDEN project - GNSS-based ATM for Rotorc...,GNSS-based ATM for Rotorcraft to Decrease Emis...


Total proyectos: 733


In [14]:
FILE_TAXONOMIA = "data/aerotax_transformado.csv"
print("FICHERO TAXONOMÍA:", FILE_TAXONOMIA)

df_tax = pd.read_csv(FILE_TAXONOMIA, encoding="utf-8")
print("Shape:", df_tax.shape)


print("Columnas:", list(df_tax.columns))

display(df_tax.head(2))

# Nos quedamos sólo con las categorías (texto completo) y su código
codes = df_tax["category_code"]

categories = df_tax["text"]
print("Número total de categorías en la taxonomía:", len(categories))

FICHERO TAXONOMÍA: data/aerotax_transformado.csv
Shape: (151, 4)
Columnas: ['category_code', 'category_label', 'parent_code', 'text']


,category_code,category_label,parent_code,text
0,1A2,Unsteady Aerodynamics,1A0,"Unsteady Aerodynamics - In aerodynamics, unste..."
1,1A11,External Noise Prediction,1A0,External Noise Prediction - Prediction of airc...


Número total de categorías en la taxonomía: 151


## 3. Cargamos el modelo de Sentence Embeddings para representar proyectos y categorias de taxonomías



In [15]:
# Si en tu entorno no está instalado.
# !pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

model = SentenceTransformer(MODEL_NAME)


emb_proj = model.encode(
    texts,
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True
)

print("Shape embeddings projects:", emb_proj.shape)

### creamos los embeddings para las categorias
emb_tax = model.encode(
    categories,
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True
)

print("Shape embeddings projects:", emb_tax.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Shape embeddings projects: (733, 768)


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Shape embeddings projects: (151, 768)


## 4. Asignamos top-k categorías con mayor similitud semántica a cada proyecto



In [17]:
import numpy as np
import pandas as pd

# emb_proj: (n_proyectos, dim)
# emb_tax:  (n_categorias, dim)
# texts: lista de textos de proyectos
# codes: lista de codes

# 1️⃣ Calculamos matriz de similitud (cosine porque están normalizados)
sims = emb_proj @ emb_tax.T  # (n_proyectos, n_categorias)

top_k = 3

# 2️⃣ Obtener índices de las top-3 categorías por proyecto
top_idx = np.argpartition(-sims, kth=top_k-1, axis=1)[:, :top_k]

# Obtener scores correspondientes
top_scores = np.take_along_axis(sims, top_idx, axis=1)

# Ordenar las top-3 por score descendente
order = np.argsort(-top_scores, axis=1)
top_idx = np.take_along_axis(top_idx, order, axis=1)
top_scores = np.take_along_axis(top_scores, order, axis=1)

# 3️⃣ Construimos el DataFrame
rows = []

for i in range(len(texts)):
    row = {
        "project_text": texts[i],
        "category_1": codes[top_idx[i, 0]],
        "score_1": float(top_scores[i, 0]),
        "category_2": codes[top_idx[i, 1]],
        "score_2": float(top_scores[i, 1]),
        "category_3": codes[top_idx[i, 2]],
        "score_3": float(top_scores[i, 2]),
    }
    rows.append(row)

df = pd.DataFrame(rows)

# 4️⃣ Guardamos en CSV
output_path = "data/project_category_assignments.csv"
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"CSV guardado en: {output_path}")
print(df.head())


CSV guardado en: data/project_category_assignments.csv
                                        project_text category_1   score_1  \
0  Tilt Rotor ATM Integrated Validation of Enviro...        1K4  0.635543   
1  GNSS-based ATM for Rotorcraft to Decrease Emis...        1D3  0.668169   
2  Curved Applications for Rotorcraft Environment...        1D3  0.592878   
3  Health Monitoring of Offshore Wind Farms. Offs...       1C11  0.567322   
4  Helicopters Deploy GNSS in Europe. The objecti...        1D3  0.628152   

  category_2   score_2 category_3   score_3  
0        1D3  0.608818        1G1  0.599349  
1        1K0  0.610545        1G5  0.601459  
2        1D4  0.578503        1E6  0.570386  
3       1D14  0.538208       1D13  0.515268  
4        1G5  0.583526        1K0  0.543773  
